# Channel & Country Performance Analysis

**Period:** July 2024 to June 2025 (12 complete months)
**Data:** `data/daily_performance.csv`
**Definitions:** RPB = revenue / bookings, Take Rate = revenue / transacted_value, ROI = revenue / spend

This notebook supports the case-study report (`report/Business_Performance_Insights.pdf`). It loads the data, computes the channel-by-country metrics required by the brief, ranks channels within each country, and produces the monthly spend chart referenced in the report.


## 1. Load and inspect the data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/daily_performance.csv')
df['booking_date'] = pd.to_datetime(df['booking_date'])

print(f"Rows: {len(df):,}")
print(f"Date range: {df['booking_date'].min().date()} to {df['booking_date'].max().date()}")
print(f"Channels: {sorted(df['channel'].unique())}")
print(f"Categories: {sorted(df['category'].unique())}")
print(f"Geographies: {sorted(df['geography'].unique())}")
df.head()


## 2. Filter to the analysis window

Past 12 complete months: **July 2024 through June 2025**.

In [ ]:
mask = (df['booking_date'] >= '2024-07-01') & (df['booking_date'] <= '2025-06-30')
d = df.loc[mask].copy()
print(f"Rows in window: {len(d):,}")


## 3. Channel x Country metrics with rankings

For each country, calculate RPB and Take Rate per channel, and rank channels within each country (1 = best).

In [ ]:
agg = d.groupby(['geography', 'channel']).agg(
    bookings=('bookings', 'sum'),
    revenue=('revenue', 'sum'),
    transacted_value=('transacted_value', 'sum'),
    spend=('spend', 'sum'),
    profit=('profit', 'sum'),
).reset_index()

agg['RPB']       = (agg['revenue'] / agg['bookings']).round(2)
agg['take_rate'] = (agg['revenue'] / agg['transacted_value'] * 100).round(2)

# Rank channels within each country (descending: 1 = best)
agg['RPB_rank']       = agg.groupby('geography')['RPB'].rank(ascending=False, method='dense').astype(int)
agg['take_rate_rank'] = agg.groupby('geography')['take_rate'].rank(ascending=False, method='dense').astype(int)

agg = agg.sort_values(['geography', 'RPB_rank']).reset_index(drop=True)
agg


## 4. Monthly spend per channel

Note: DIRECT and SEO are organic channels with effectively zero spend across the 12-month window. Plotting them on the same axis as SEM and META compresses the meaningful data and creates misleading spikes from rounding noise. The chart below plots only the channels with material spend (SEM and META) and notes the organic channels in the caption.

In [ ]:
d['month'] = d['booking_date'].dt.to_period('M').dt.to_timestamp()
spend_total = d.groupby('channel')['spend'].sum()
print("Total 12-month spend by channel (EUR):")
print(spend_total.round(0))


In [ ]:
monthly = (
    d[d['channel'].isin(['SEM', 'META'])]
    .groupby(['month', 'channel'])['spend'].sum()
    .unstack('channel')
    .sort_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(monthly.index, monthly['SEM'] / 1e6, marker='o', label='SEM', linewidth=2)
ax.plot(monthly.index, monthly['META'] / 1e6, marker='o', label='META', linewidth=2)

ax.set_title('Monthly Spend by Channel (SEM and META)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Spend (EUR, millions)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.text(0.5, -0.02,
         'DIRECT and SEO omitted: organic channels with under EUR 1K total spend across the 12-month window.',
         ha='center', fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.savefig('../report/monthly_spend_by_channel.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Key insights extracted

Detailed analysis lives in `report/Business_Performance_Insights.pdf`. The headline points:

1. **HOTEL is the most underused product.** Highest margin (59.1%) and ROI (2.44x) but only 9.3% of profit.
2. **FLIGHT + HOTEL is the profit engine.** 70.5% of profit on 20.6% of bookings, RPB EUR 190.
3. **OTHER geography needs restructuring.** 32.9% of bookings but worst margin (31.6%) and 68% cost ratio.
4. **June SEM blew up.** Spend +50.8% drove revenue only +18.7%, cutting margin from 50.5% to 37.1%.
5. **Organic channels carry the business.** DIRECT + SEO = 63.8% of profit on near-zero spend.
